In [1]:
!pip install torch torchvision

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms,models
from torchvision.models import MobileNet_V2_Weights
from torch.utils.data import DataLoader
import os

In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
data_transforms ={
    "Training":transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0))
        ], p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    "Testing":
    transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
}

In [5]:
dir = "Dataset/BT-MRI Dataset/BT-MRI Dataset"

In [6]:
image_data = {
    x: datasets.ImageFolder(os.path.join(dir, x), data_transforms[x])
    for x in ["Training", "Testing"]
}

In [7]:
print("Successfully loaded datasets!")
print(f"Training images: {len(image_data['Training'])}")
print(f"Testing images: {len(image_data['Testing'])}")

Successfully loaded datasets!
Training images: 5712
Testing images: 1311


In [8]:
dataloaders = {
    x: DataLoader(image_data[x], batch_size=32, shuffle=True, num_workers=4)
    for x in ["Training", "Testing"]
}

In [9]:
model = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

In [10]:
for params in model.parameters():
    params.requires_grad=False

In [11]:
num_classes = 4
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

In [12]:
loss_ = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr=0.001)
epochs = 5

In [13]:
for param in model.classifier.parameters():
    param.requires_grad = True

# Check what device PyTorch thinks it's using
print(f"Current Device: {device}")

# Check where the model actually lives
print(f"Model is on: {next(model.parameters()).device}")

for i in range(epochs+1):
    model.train()
    running_loss=0.0
    for image,label in dataloaders["Training"]:
        image,label = image.to(device),label.to(device)
        optimizer.zero_grad()

        output = model(image)
        loss = loss_(output,label)
        loss.backward()
        optimizer.step()

        running_loss +=loss.item()*image.size(0)
    epoch_loss = running_loss/len(image_data["Training"])
    print(f"Epoch{i+1}/{epochs} - Loss {epoch_loss:.4f}")

Current Device: cuda
Model is on: cuda:0
Epoch1/5 - Loss 0.9866
Epoch2/5 - Loss 0.7619
Epoch3/5 - Loss 0.6937
Epoch4/5 - Loss 0.6691
Epoch5/5 - Loss 0.6703
Epoch6/5 - Loss 0.6527


In [17]:
for i in model.parameters():
    i.requires_grad=True

In [24]:
optimizer_ft = optim.Adam(model.classifier.parameters(),lr=1e-5)
epochs_ft = 10

In [25]:
print(f"Current Device: {device}")
print(f"Model is on: {next(model.parameters()).device}")
for epoch in range(epochs_ft):
    model.train()
    running_loss = 0.0
    
    for image, label in dataloaders["Training"]:
        image, label = image.to(device), label.to(device)
        
        optimizer_ft.zero_grad()
        outputs = model(image)
        loss = loss_(outputs, label)
        loss.backward()
        optimizer_ft.step()
        
        running_loss += loss.item() * image.size(0)
        
    epoch_loss = running_loss / len(image_data["Training"])
    print(f'Fine-Tune Epoch {epoch+1}/{epochs_ft} - Loss: {epoch_loss:.4f}')

Current Device: cuda
Model is on: cuda:0
Fine-Tune Epoch 1/10 - Loss: 0.6169
Fine-Tune Epoch 2/10 - Loss: 0.6038
Fine-Tune Epoch 3/10 - Loss: 0.6056
Fine-Tune Epoch 4/10 - Loss: 0.6116
Fine-Tune Epoch 5/10 - Loss: 0.6019
Fine-Tune Epoch 6/10 - Loss: 0.6200
Fine-Tune Epoch 7/10 - Loss: 0.6185
Fine-Tune Epoch 8/10 - Loss: 0.6026
Fine-Tune Epoch 9/10 - Loss: 0.6181
Fine-Tune Epoch 10/10 - Loss: 0.6279


In [26]:
model.eval()
correct = 0
total = 0
print("Evaluating on the standard Testing dataset...")
with torch.no_grad():
    for images, labels in dataloaders["Testing"]:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on clean Testing data: {accuracy:.2f}%')

Evaluating on the standard Testing dataset...
Accuracy on clean Testing data: 76.96%


In [28]:
import os
from torch.utils.data import DataLoader

challenging_base_dir = "Dataset/Challenging Datasets/Challenging Datasets"

challenging_folders = [
    "Blurred Dataset", 
    "Noisy Dataset", 
    "Patient Motion Artifact Dataset"
]

print("--- Starting Stress Test on Challenging Data ---")

for folder_name in challenging_folders:
    folder_path = os.path.join(challenging_base_dir, folder_name)
    test_dataset = datasets.ImageFolder(folder_path, data_transforms['Testing'])
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    print(f"Accuracy on {folder_name}: {accuracy:.2f}%")

--- Starting Stress Test on Challenging Data ---
Accuracy on Blurred Dataset: 74.96%
Accuracy on Noisy Dataset: 32.62%
Accuracy on Patient Motion Artifact Dataset: 34.17%
